# NB5 · MSC-KD — the method (Q5)

**Only run this after Q1–Q4 are in.** The method is the last section of the
paper, not its thesis: Q1, Q2 and Q3 are publishable whichever way this goes.

Distils the teacher's per-sample compute requirement into a student's monotone
routing policy. Three loss terms, two weights:

    L = L_CE + α·L_KD + β·L_MSC

Monotonicity is architectural, not a penalty — the sufficiency head is a
cumulative-link ordinal head whose thresholds are `θ_{k+1} = θ_k + softplus(δ_k)`,
so the predicted curve is non-decreasing in k **by construction**. A constraint
that cannot be violated beats a soft penalty that can trade off against other
terms.

## Both arms run in one pass

The **scrambled control** (MSC targets permuted within the batch) trains first.
If it matches the real arm, `L_MSC` is a regulariser and not a signal, and you
need to know that before writing anything.

This used to be a module-level flag with a comment saying which value to run
first. The flag defaulted to the control, four sessions in a row trained the
control, and the real arm never existed. **An invariant in a comment is not a
mechanism** — so both arms are a loop now, and whether a run is scrambled is
derived from its own `run_id`.

## The budget count comes from the student, never the teacher

A student's usable exits are adaptive: `resnet18` and `resnet50` do not have the
same number. Sizing the router from the *teacher's* budget grid produces a model
that trains fine — the loss only ever compares the head against targets, both on
the teacher's grid — and then fails at *evaluation*, where routing indexes the
student's actual exits. It is a modelling error, not a shape bug: the routing
decision spends **the student's** compute, so the teacher's grid is meaningless.

That defect took six rounds to fix because it was patched one call site at a
time, and one of those rounds recreated it *inside the dry run written to catch
it*.

In [ ]:
# ============================================================================
# CELL 1 -- unpack the library.  Runs in every notebook.  No network.
# ============================================================================
# Writes two files into the working directory and imports them:
#
#   msc_lib.py    cbeb9a5b2196   the pipeline: data, zoo, training, measurement
#   msc_core.py   2cc4ba5e0935   the reference maths: the MSC definition and
#                                    every statistic in the paper
#
# Both are GENERATED from src/ by build_notebooks_in100.py. Editing the base64
# below does nothing that survives a rebuild -- edit src/msc_lib.py instead.
#
# NOTHING IS INSTALLED HERE. This pipeline runs offline; the packages must
# already be present (see requirements.txt). A missing one is reported by name
# with what it costs you, rather than silently pip-installing on a machine that
# may have no network.
import base64, os, sys
from pathlib import Path

# Offline guards must be set BEFORE anything that might fetch is imported.
os.environ.setdefault('MSC_OFFLINE', '1')

WORK = Path.cwd()
_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]          # force reimport if this cell is re-run

_MISSING = []
for _pkg, _why in (('torch', 'everything'),
                   ('torchvision', 'resnet/vgg/shufflenet/swin'),
                   ('numpy', 'everything'), ('pandas', 'every table'),
                   ('pyarrow', 'per_sample/*.parquet -- the science'),
                   ('yaml', 'config.yaml per run'),
                   ('scipy', 'Spearman = Q1 and Q3'),
                   ('sklearn', 'Q4 delta-R2, Q2 PCA'),
                   ('psutil', 'host telemetry columns'),
                   ('pynvml', 'GPU power -- energy columns are NA without it'),
                   ('fvcore', 'FLOPs. rho is DEFINED in FLOPs.')):
    try:
        __import__(_pkg)
    except ImportError:
        _MISSING.append(f'{_pkg:12s} {_why}')
if _MISSING:
    print('MISSING PACKAGES -- install these, then restart the kernel:')
    for _m in _MISSING:
        print('   ', _m)
    raise SystemExit('see requirements.txt')

import msc_lib as M
import torch

print(f'msc_lib {M.__version__}   torch {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  GPU {_i}: {_p.name}  {_p.total_memory/2**30:.1f} GiB  sm_{_p.major}{_p.minor}')
else:
    print('  *** NO CUDA. A CPU-only torch trains at roughly 1/200th speed')
    print('  *** while reporting entirely plausible numbers. Fix this first.')

In [ ]:
# ============================================================================
# CELL 2 -- WHERE EVERYTHING LIVES
# ============================================================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine gets `msc_data/in100` and `msc_results`.
#
# The previous version defaulted to r'D:\msc_data\in100'. There is no D:
# drive here, and the failure was
#
#     FileNotFoundError: [WinError 3] The system cannot find the path
#     specified: 'D:\'
#
# forty lines deep inside pathlib, naming neither the setting nor the file that
# had to change. A default that names a drive letter is wrong on any machine
# without that letter (D-44).
#
# Set them explicitly if you want somewhere specific. Both are checked below by
# WRITING A PROBE FILE AND READING IT BACK -- os.access lies on Windows shares.
#
#   data     ~26 GB   the packed dataset, read-only after NB1
#   results ~120 GB   every run. Nothing here is ever deleted.

DATA_DIR = None      # e.g. r'E:\msc_data\in100'   -- None = choose for me
MSC_ROOT = None      # e.g. r'E:\msc_results'        -- None = choose for me

# ---------------------------------------------------------------------------
import os

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_IN100_DIR'] = DATA_DIR
os.environ['MSC_SCRATCH'] = MSC_ROOT

sess = M.Session(account='local', phase='p3', dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 worker_id=0, num_workers=1)

print()
print('layout under MSC_ROOT:')
print('  runs/{run_id}/  config.yaml  summary.json  STATUS.json')
print('                   metrics/     epochs.csv  final.csv  confusion_matrix.csv')
print('                                per_class.csv  exit_metrics.csv')
print('                   telemetry/   energy_samples.csv  system_samples.csv')
print('                                step_traces.jsonl')
print('                   per_sample/  test.parquet  train_holdout.parquet')
print('                                train_dynamics.parquet  meta.json')
print('                   checkpoints/ ckpt_last.pt  ckpt_best.pt')
print('                   env/         environment.json')
print('                   exit_heads.pt')
print('  budgets/{arch}.json     FLOPs per compute configuration')
print('  registry/events/*.jsonl  what ran, when, and how it ended')
print('  analysis/                Q1-Q4 outputs')
print('  tables/  paper/figures/  console/')

In [ ]:
TEACHER  = 'resnet50'
STUDENTS = ['resnet18', 'shufflenetv2_in', 'deit_small']
SEEDS    = (1, 2, 3)
ARMS     = [True, False]          # control FIRST, so a null result stops you early

sess = M.Session(account='local', phase='p3', dataset='imagenet100',
                 work_root=MSC_ROOT, session_limit_h=0.0)

t_runs = [r['run_id'] for r in sess.completed_runs(phase='p1')
          if M.parse_run_id(r['run_id'])['arch'] == TEACHER
          and sess.measured(r['run_id'])]
if not t_runs:
    raise SystemExit(f'no measured {TEACHER} run. Run NB2 and NB3 first.')
teacher_run = sorted(t_runs)[0]
print(f'teacher: {teacher_run}')

cfgs = []
for shuffled in ARMS:
    for a in STUDENTS:
        for s in SEEDS:
            method = ('mscKDshuffrom' if shuffled else 'mscKDfrom') + TEACHER
            cfgs.append(sess.config(a, seed=s, method=method,
                                    teacher_run=teacher_run,
                                    shuffle_msc_targets=bool(shuffled)))
print(f'{len(cfgs)} student run(s): {len(STUDENTS)} arch x {len(SEEDS)} seeds x 2 arms')

In [ ]:
results = sess.run_all(M.train_msc_kd, cfgs)
real = [r for r in results if 'shuff' not in r['run_id']]
print(f"\n{len([r for r in real if r.get('status') != 'skipped'])}/"
      f"{len(real)} REAL-method students trained -- the comparison needs all of them")

---
## Compare at matched FLOPs

The only comparison that means anything. B2 (confidence-threshold routing) is
where the field actually is; B11 (routing by the student's own true post-hoc
MSC) is the ceiling. **The fraction of the B2→B11 gap that MSC-KD closes is the
result.**

`γ` is calibrated by Learn-then-Test on `train_holdout`. At 15,000 samples the
distribution-free guarantee holds at **ε = 0.01** — CIFAR had only 5,000 and had
to settle for ε = 0.03 and say so in its limitations.

In [ ]:
cmp_ = M.compare_routing_methods(sess, run_ids=[r['run_id'] for r in results])
M.save_analysis(sess.data_dir, 'q5_method_comparison', cmp_)
display(cmp_)

In [ ]:
status = sess.confirm_on_disk([r['run_id'] for r in results])
print()
print('Then NB4 for the final tables, and check that the SCRAMBLED arm is')
print('clearly worse than the real one. If it is not, L_MSC is a regulariser')
print('and the mechanism claim is wrong even if the method wins.')